# EDA and Model Training (KNN and Logistic Regression)

## Loading the Dataset
Loading data from csv and checking dimensions and quality of data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


df_raw = pd.read_csv("data/emp.csv")
df_raw.head()

In [ ]:
df_raw.tail()

In [ ]:
df_raw.info()

Checking number of null and missing values in each column

In [ ]:
df_raw.isna().sum()

In [ ]:
df_raw.isnull().sum()

Next, Checking which one of these text columns are categorical and how many values they have to determine if one hot encoding would be viable.

In [ ]:
list_of_text_cols = df_raw.select_dtypes(include="str").columns.tolist()

for col in list_of_text_cols:
    print(df_raw[col].value_counts() , "\n")



BusinessTravel and MaritialStatus can be Label encoded as the categories are related to each other. JobRole can be label encoded as it has too many categories and one hot encoding can unneceserily increase compute (Can experiment with One-Hot and Label encoding both to determine if increasing dimensions and extra compute is worth it.)  \
Over18, StandardHours, EmployeeNumber, EmployeeCount should be removed to reduce compute as they have no valuable data that could be used.  \
Rest can be converted with one hot encoding for training.

In [ ]:
df_raw = df_raw.drop(columns=["Over18", "StandardHours", "EmployeeNumber", "EmployeeCount"])

In [ ]:
df_raw.info()

## Splitting data

Splitting data before checking linear correlation and visual non linear patterns to reduce Selection bias and Optimistic bias.

In [ ]:
from sklearn.model_selection import train_test_split

df_train : pd.DataFrame
df_test : pd.DataFrame

df_train, df_test = train_test_split(df_raw, test_size=0.2, stratify=df_raw["Attrition"], random_state=42)

Chose to do stratified split to maintain the target class ratio, as there was class imbalance in our target feature. Checking class ratio in both sets below:

In [ ]:
df_train["Attrition"].value_counts()

In [ ]:
986/190

In [ ]:
df_test["Attrition"].value_counts()

In [ ]:
247/47

In [ ]:
df_train.head()

## EDA and Visual Patterns

In [ ]:

df_train.hist(bins=50, figsize=(20,20))

In [ ]:
import seaborn as sns

sns.pairplot(df_train, hue="Attrition")

In [ ]:
sns.countplot(df_train, x="Attrition", hue="Attrition")

In [ ]:
sns.pairplot(
    df_train[["Age", "MonthlyIncome", "DistanceFromHome", "Attrition"]],
    hue="Attrition",
    palette="plasma"
)

### EDA of relevant numerical values against Attrition

In [ ]:
sns.boxplot(
    data=df_train,
    x="Attrition",
    y="Age",
    hue="Attrition",
)

plt.show()

In [ ]:
sns.histplot(x="DistanceFromHome", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.boxplot(x="Attrition", y="EnvironmentSatisfaction", hue="Attrition", data=df_train)
plt.show()

In [ ]:
sns.histplot(x="HourlyRate", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.histplot(x="JobSatisfaction", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.histplot(x="JobInvolvement", hue="Attrition", data=df_train, stat="density", common_norm=False)
plt.show()

In [ ]:
sns.boxplot(x="Attrition", y="MonthlyIncome", data=df_train, hue="Attrition")
plt.plot()

In [ ]:
sns.histplot(x="MonthlyIncome", data=df_train, hue="Attrition", common_norm=False, stat="density")

In [ ]:
sns.histplot(x="YearsAtCompany", data=df_train, hue="Attrition", common_norm=False, stat="density")

In [ ]:
sns.boxplot(x="Attrition", y="YearsAtCompany", data=df_train, hue="Attrition")

In [ ]:
sns.histplot(x="YearsInCurrentRole", data=df_train, hue="Attrition", common_norm=False, stat="density")

In [ ]:
sns.boxplot(x="Attrition", y="YearsInCurrentRole", data=df_train, hue="Attrition",)

In [ ]:
sns.histplot(x="YearsSinceLastPromotion", data=df_train, hue="Attrition", common_norm=False, stat="density")

### EDA of relevant categorical values against Attrition

In [ ]:
df_train.info()

In [ ]:
pd.crosstab(df_train["Department"], df_train["Attrition"], normalize="index").plot(kind="bar", stacked=True)
plt.plot()

In [ ]:
sns.countplot(x="Department", hue="Attrition", data=df_train,)

In [ ]:
pd.crosstab(df_train["Gender"], df_train["Attrition"], normalize="index").plot(kind="bar")

In [ ]:
pd.crosstab(df_train["BusinessTravel"], df_train["Attrition"], normalize="index").plot(kind="bar")
plt.plot()

In [ ]:
pd.crosstab(df_train["JobRole"], df_train["Attrition"], normalize="index").plot(kind="bar")
plt.plot()

In [ ]:
pd.crosstab(df_train["MaritalStatus"], df_train["Attrition"], normalize="index").plot(kind="bar")

In [ ]:
pd.crosstab(df_train["OverTime"], df_train["Attrition"], normalize="index").plot(kind="bar",)
plt.plot()

EDA of various numerical and categorical values prove that most of the features have some effect on attrition and most of the features have valuable information that our model will need. Only column that will be removed is Over18 because it has same value across the dataset.

## Creating training pipeline

Creating a training pipeline with imputing, scaling and encoding. So this single pipeline could be used for multiple experiments.

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder =LabelEncoder()

df_train_x = df_train.drop(columns=["Attrition"])
df_train_y = np.asarray(label_encoder.fit_transform(df_train["Attrition"]))

df_test_x = df_test.drop(columns=["Attrition"])
df_test_y = np.asarray(label_encoder.transform(df_test["Attrition"]))



numerical_cols = df_train.select_dtypes(include="int64").columns.tolist()


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder

simple_imputer_num = SimpleImputer(strategy="median")
simple_imputer_cat = SimpleImputer(strategy="most_frequent")
standard_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder()
ordinal_encoder = OrdinalEncoder()

cat_onehot_pipeline = Pipeline([("simpleImputerCat", simple_imputer_cat), ("oneHotEncoder", one_hot_encoder)])
cat_label_pipeline = Pipeline([("simpleImputerCat", simple_imputer_cat), ("labelEncoder", ordinal_encoder)])
num_pipeline = Pipeline([("simpleImputerNum", simple_imputer_num), ("standardScaler", standard_scaler)])

final_model = ColumnTransformer([("catLabel", cat_label_pipeline, ["BusinessTravel" ,"JobRole", "MaritalStatus"]),
                                 ("catOneHot", cat_onehot_pipeline, ["OverTime", "Gender", "Department", "EducationField"]),
                                 ("num", num_pipeline, numerical_cols)])




## Training Model

### Finding optimal no. of K

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

final_pipeline = Pipeline([("model",final_model), ("knn", KNeighborsClassifier())])



In [ ]:
from sklearn.metrics import f1_score

training_errors =[]
test_errors = []

k_values = range(1,31)

for k in k_values:
    final_pipeline.set_params(knn__n_neighbors=k)
    final_pipeline.fit(df_train_x, df_train_y)
    
    train_pred = final_pipeline.predict(df_train_x)
    test_pred = final_pipeline.predict(df_test_x)
    
    train_accuracy = f1_score(df_train_y, train_pred)
    test_accuracy = f1_score(df_test_y, test_pred)
    
    training_errors.append(1-train_accuracy)
    test_errors.append(1-test_accuracy)
    
plt.figure(figsize=(14,8))
plt.plot(k_values, training_errors, marker="o", label="Training error")
plt.plot(k_values, test_errors, marker="o", label="Test error")
plt.title("Training & Test Errors vs No. of K")
plt.xlabel("No. of neighbors")
plt.ylabel("F1")
plt.grid(True)
plt.xticks(k_values)
plt.show()
    
    

Most optimal k value according to the graph seems to be 9, as error somewhat stabilizes after that. I can go with 17 as well, but that would increase a lot of compute for very marginal improvement. So deciding with k=9 as optimal for now.

### Evaluating with K-Fold cross val on training set

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(knn__n_neighbors=3)

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))


In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

### Evaluating on test set

In [ ]:
final_pipeline.set_params(knn__n_neighbors=3)

final_pipeline.fit(df_train_x, df_train_y)

pred = final_pipeline.predict(df_test_x)

confusion_mat = confusion_matrix(df_test_y, pred)



print("Evaluation on test data:")
print(classification_report(df_test_y, pred))

In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.plot()

## Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(final_pipeline, param_grid=[{"knn__n_neighbors": [3,5,7,9,11,13,15,17]}], scoring="f1")

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)

In [ ]:
grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski", "manhattan", "euclidean"],
    "knn__leaf_size": [15, 30, 45, 60],
    "knn__algorithm": ["auto", "ball_tree", "kd_tree", "brute"]
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="f1" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

Hence, after above tuning of hyperparameters I will select the above configuration.

## Evaluating on test set after Hyperparameter Tuning

In [ ]:
final_pipeline.set_params(knn__n_neighbors=3, knn__algorithm="auto", knn__leaf_size=15, knn__metric="minkowski", knn__weights="uniform")

final_pipeline.fit(df_train_x, df_train_y)

pred = final_pipeline.predict(df_test_x)

confusion_mat = confusion_matrix(df_test_y, pred)



print("K-fold Evaluation on test data:")
print(classification_report(df_test_y, pred))

In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.plot()

## Conclusion

Apparently, hyperparameter tuning did not have much impact on the model's performance. Despite achieving good accuracy, the model's overall performance is still mediocre, and it seems to be severely underfitting in terms of the problem we're trying to solve (identifying employees likely to leave).

It also seems that KNN might not be the right choice for this kind of dataset, and there are a few reasons behind that:

1. Severe class imbalance: There is a significant imbalance in our target variable, "Attrition", with considerably more employees staying than leaving.

2. Lack of strong patterns: We couldn't identify strong patterns across the dataset. The data appears to be scattered, with considerable overlap between the two target classes.

3. High dimensionality: The dataset has a relatively large number of features, and almost none of them has a strong relationship with "Attrition". Most features seem to have only a weak individual relationship with the target variable.

These factors make it difficult for KNN to identify meaningful similarities between employees and accurately classify those who are likely to leave.

But it does not mean that this dataset and problem is hopeless, as the assignment ends here, I am ending my experimentation here. If i were to explore further, class imbalance reduction with SMOTE can be explored and experimented with. Also other models can also be experimented with, with and without SMOTE, Logistic regression, Random forest classifier and SVM can be experimented with this data and problemset.

## Experimenting with SMOTE (Not part of assignment from here)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, OrdinalEncoder
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline as ImbPipeline

cat_cols = ["BusinessTravel" ,"JobRole", "MaritalStatus", "OverTime", "Gender", "Department", "EducationField"]

simple_imputer_num = SimpleImputer(strategy="median")
simple_imputer_cat = SimpleImputer(strategy="most_frequent")
standard_scaler = StandardScaler()
one_hot_encoder = OneHotEncoder()
ordinal_encoder = OrdinalEncoder()

cat_pipeline = Pipeline([("simpleImputerCat", simple_imputer_cat)])
num_pipeline = Pipeline([("simpleImputerNum", simple_imputer_num), ("standardScaler", standard_scaler)])

onehot_pipeline = Pipeline([ ("oneHotEncoder", one_hot_encoder)])
# ordinal_pipeline = Pipeline([("labelEncoder", ordinal_encoder)])

preprocessing_pipeline = ColumnTransformer([("num", num_pipeline, numerical_cols),
                                            ("cat", cat_pipeline, cat_cols),
                                            ])

encoding_pipeline = ColumnTransformer([("num", "passthrough", list(range(len(numerical_cols)))),
                                       ("onehot", one_hot_encoder, list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols)))),
                                      ])


final_pipeline = ImbPipeline([("preprocessing", preprocessing_pipeline),
                              ("smote", SMOTENC(categorical_features=list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols))), sampling_strategy="auto", k_neighbors=5, random_state=42)), 
                              ("encoding", encoding_pipeline),
                              ("knn", KNeighborsClassifier())])




### Using f1 to plot different values of k and their errors

In [ ]:
from sklearn.metrics import f1_score

training_errors =[]
test_errors = []

k_values = range(1,31, 2)

for k in k_values:
    final_pipeline.set_params(knn__n_neighbors=k)
    final_pipeline.fit(df_train_x, df_train_y)
    
    train_pred = final_pipeline.predict(df_train_x)
    test_pred = final_pipeline.predict(df_test_x)
    
    train_accuracy = f1_score(df_train_y, train_pred)
    test_accuracy = f1_score(df_test_y, test_pred)
    
    training_errors.append(1-train_accuracy)
    test_errors.append(1-test_accuracy)
    
plt.figure(figsize=(14,8))
plt.plot(k_values, training_errors, marker="o", label="Training error")
plt.plot(k_values, test_errors, marker="o", label="Test error")
plt.title("Training & Test Errors vs No. of K")
plt.xlabel("No. of neighbors")
plt.ylabel("F1")
plt.grid(True)
plt.xticks(k_values)
plt.show()
    
    

### Using recall to plot different values of k and their errors

In [ ]:
from sklearn.metrics import recall_score

training_errors =[]
test_errors = []

k_values = range(1,31, 2)

for k in k_values:
    final_pipeline.set_params(knn__n_neighbors=k)
    final_pipeline.fit(df_train_x, df_train_y)
    
    train_pred = final_pipeline.predict(df_train_x)
    test_pred = final_pipeline.predict(df_test_x)
    
    train_accuracy =recall_score(df_train_y, train_pred)
    test_accuracy = recall_score(df_test_y, test_pred)
    
    training_errors.append(1-train_accuracy)
    test_errors.append(1-test_accuracy)
    
plt.figure(figsize=(14,8))
plt.plot(k_values, training_errors, marker="o", label="Training error")
plt.plot(k_values, test_errors, marker="o", label="Test error")
plt.title("Training & Test Errors vs No. of K")
plt.xlabel("No. of neighbors")
plt.ylabel("Recall")
plt.grid(True)
plt.xticks(k_values)
plt.show()
    
    

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(knn__algorithm= 'auto', knn__leaf_size= 30, knn__metric= 'manhattan', knn__n_neighbors= 13, knn__weights= 'distance')
#after experimentation chose 13 kn neighbors as optimal, with good precision and recall for minority class

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))


In [ ]:
import seaborn as sns 

sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

### Trying smaller k, but recall drops significantly for minority

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(knn__n_neighbors=3)

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))


In [ ]:
import seaborn as sns 

sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [ ]:
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(final_pipeline, param_grid=[{"knn__n_neighbors": [3,5,7,9,11,13,15,17]}], scoring="f1")

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)

In [ ]:
grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15, 21, 31],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["minkowski", "manhattan", "euclidean"],
    "knn__leaf_size": [15, 30, 45, 60],
    "knn__algorithm": ["auto", "ball_tree", "kd_tree", "brute"]
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="f1" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(smote__sampling_strategy=1.0 ,knn__algorithm= 'auto', knn__leaf_size= 30, knn__metric= 'manhattan', knn__n_neighbors= 13, knn__weights= 'distance')

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))


In [ ]:
import seaborn as sns 

sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

Selecting the above config as it produces the best results uptill now, with precision ~0.36 and recall ~0.63 for minority class, also gives good numbers with majority class.

## Eval on test set after HP Tuning with SMOTE

In [ ]:

final_pipeline.fit(df_train_x, df_train_y)

pred = final_pipeline.predict(df_test_x)

confusion_mat = confusion_matrix(df_test_y, pred)



print("Evaluation on test data:")
print(classification_report(df_test_y, pred))

In [ ]:
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')

As we can see, SMOTE improved recall for minority but the tradeoff is precision for minority, no of false positives in majority class has increased as a tradeoff, the model is improved to a significant degree than before but KNN is definitely struggling on this data because of the previously mentioned observations of this data, it would be fascinating to explore other models on this data with smote.

## Using random forest classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

final_pipeline = ImbPipeline([("preprocessing", preprocessing_pipeline),
                              ("smote", SMOTENC(categorical_features=list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols))), sampling_strategy="auto", k_neighbors=5, random_state=42)), 
                              ("encoding", encoding_pipeline),
                              ("rf", RandomForestClassifier(max_depth=None, n_estimators=100))])

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report


kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))


In [ ]:
from sklearn.model_selection import GridSearchCV


grid = {
    "rf__n_estimators": [100, 200, 300, 500],
    "rf__max_depth": [None, 5, 10, 15, 20, 30],
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="recall" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

## Using Logistic regression

In [ ]:
from sklearn.linear_model import LogisticRegression

final_pipeline = ImbPipeline([("preprocessing", preprocessing_pipeline),
                              ("smote", SMOTENC(categorical_features=list(range(len(numerical_cols), len(numerical_cols)+len(cat_cols))), sampling_strategy="auto", k_neighbors=5, random_state=42)), 
                              ("encoding", encoding_pipeline),
                              ("lr", LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
))])

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(smote__sampling_strategy=1.0)

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))


In [ ]:
from sklearn.model_selection import GridSearchCV


grid = {
    "lr__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "lr__penalty": ["l1", "l2"],
    # "lr__solver": ["lbfgs", "liblinear", "saga"],
    # "lr__class_weight": [None, "balanced"],
    # "lr__fit_intercept": [True, False],
    "lr__max_iter": [1000, 2000, 5000],
    # "lr__l1_ratio": []
    
}

gs = GridSearchCV(final_pipeline, param_grid=grid, scoring="f1" , verbose=1, n_jobs=1, cv=5)

gs.fit(df_train_x, df_train_y)

print(gs.best_params_)
print(gs.best_score_)

In [ ]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

final_pipeline.set_params(smote__sampling_strategy=1.0, lr__C=  0.1, lr__max_iter=  1000, lr__penalty= 'l2')

kf = KFold(n_splits=5)

pred = cross_val_predict(final_pipeline, df_train_x, df_train_y)

confusion_mat = confusion_matrix(df_train_y, pred)



print("K-fold Evaluation on training data:")
print(classification_report(df_train_y, pred))


In [ ]:
import seaborn as sns
sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', )
plt.xlabel('Predicted')
plt.ylabel('Actual')